In [1]:
import re
from typing import List, Dict, Any
import logging
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

class GoogleSheetsManager:
	"""
	A class to manage interactions with Google Sheets.
	Handles authentication, reading, and writing operations.
	"""
	
	def __init__(self, credentials_file: str):
		"""
		Initialize the Google Sheets manager with credentials.
		
		Args:
			credentials_file (str): Path to the service account credentials JSON file
		"""
		self.logger = logging.getLogger(__name__)
		
		# Setup authentication
		scopes = ['https://www.googleapis.com/auth/spreadsheets']
		try:
			self.creds = service_account.Credentials.from_service_account_file(
				credentials_file, scopes=scopes)
			self.service = build('sheets', 'v4', credentials=self.creds)
			self.sheets = self.service.spreadsheets()
			self.logger.info("Successfully authenticated with Google Sheets API")
		except Exception as e:
			self.logger.error(f"Failed to initialize Google Sheets service: {e}")
			raise

	def extract_spreadsheet_id(self, sheet_url: str) -> str:
		"""
		Extract the spreadsheet ID from a Google Sheets URL.
		
		Args:
			sheet_url (str): The URL of the Google Sheet
			
		Returns:
			str: The extracted spreadsheet ID
			
		Raises:
			ValueError: If the URL is invalid or ID cannot be extracted
		"""
		pattern = r'/spreadsheets/d/([a-zA-Z0-9-_]+)'
		match = re.search(pattern, sheet_url)
		if match:
			spreadsheet_id = match.group(1)
			self.logger.debug(f"Extracted spreadsheet ID: {spreadsheet_id}")
			return spreadsheet_id
		raise ValueError("Invalid Google Sheet URL, could not extract spreadsheet ID")

	def get_sheet_id(self, spreadsheet_id: str, sheet_name: str) -> int:
		"""
		Get the internal sheet ID for a named sheet within a spreadsheet.
		
		Args:
			spreadsheet_id (str): The ID of the spreadsheet
			sheet_name (str): The name of the sheet
			
		Returns:
			int: The internal sheet ID
			
		Raises:
			ValueError: If the sheet name doesn't exist in the spreadsheet
		"""
		try:
			spreadsheet = self.sheets.get(spreadsheetId=spreadsheet_id).execute()
			sheets = spreadsheet.get('sheets', [])
			
			for sheet in sheets:
				if sheet['properties']['title'] == sheet_name:
					sheet_id = sheet['properties']['sheetId']
					self.logger.debug(f"Found sheet '{sheet_name}' with ID: {sheet_id}")
					return sheet_id
					
			available_sheets = [sheet['properties']['title'] for sheet in sheets]
			raise ValueError(f"Sheet '{sheet_name}' not found. Available sheets: {available_sheets}")
			
		except HttpError as e:
			self.logger.error(f"API error when getting sheet ID: {e}")
			raise
		except Exception as e:
			self.logger.error(f"Error getting sheet ID: {e}")
			raise

	def read_range(self, spreadsheet_id: str, sheet_name: str, 
					range_notation: str) -> List[List[Any]]:
		"""
		Read data from a specific range in a sheet.
		
		Args:
			spreadsheet_id (str): The ID of the spreadsheet
			sheet_name (str): Name of the sheet to read from
			range_notation (str): A1 notation range (e.g., 'A1:C10')
			
		Returns:
			List[List[Any]]: The data read from the sheet
		"""
		full_range = f"'{sheet_name}'!{range_notation}"
		try:
			result = self.sheets.values().get(
				spreadsheetId=spreadsheet_id,
				range=full_range
			).execute()
			
			values = result.get('values', [])
			self.logger.info(f"Read {len(values)} rows from {full_range}")
			return values
			
		except HttpError as e:
			self.logger.error(f"API error when reading {full_range}: {e}")
			raise
		except Exception as e:
			self.logger.error(f"Error reading {full_range}: {e}")
			raise

	def read_column(self, spreadsheet_id: str, sheet_name: str, 
					 column: str, start_row: int = 1) -> List[Any]:
		"""
		Read a specific column from a sheet.
		
		Args:
			spreadsheet_id (str): The ID of the spreadsheet
			sheet_name (str): Name of the sheet to read from
			column (str): Column letter (e.g., 'A')
			start_row (int): Starting row (1-indexed)
			
		Returns:
			List[Any]: Values from the column
		"""
		range_name = f"'{sheet_name}'!{column}{start_row}:{column}"
		try:
			result = self.sheets.values().get(
				spreadsheetId=spreadsheet_id,
				range=range_name
			).execute()
			
			values = result.get('values', [])
			# Flatten the list of lists into a single list
			flattened = [row[0] if row else "" for row in values]
			
			self.logger.info(f"Read {len(flattened)} values from column {column}")
			return flattened
			
		except HttpError as e:
			self.logger.error(f"API error when reading column {column}: {e}")
			raise
		except Exception as e:
			self.logger.error(f"Error reading column {column}: {e}")
			raise

	def read_row_data(self, spreadsheet_id: str, sheet_name: str, 
					 row_number: int, start_col: str = 'A', end_col: str = 'Z') -> List[Any]:
		"""
		Read data from a specific row.
		
		Args:
			spreadsheet_id (str): The ID of the spreadsheet
			sheet_name (str): Name of the sheet to read from
			row_number (int): Row number (1-indexed)
			start_col (str): Starting column letter
			end_col (str): Ending column letter
			
		Returns:
			List[Any]: Values from the row
		"""
		range_name = f"'{sheet_name}'!{start_col}{row_number}:{end_col}{row_number}"
		try:
			result = self.sheets.values().get(
				spreadsheetId=spreadsheet_id,
				range=range_name
			).execute()
			
			values = result.get('values', [])
			# Return the first row if it exists, otherwise an empty list
			row_data = values[0] if values else []
			
			self.logger.info(f"Read data from row {row_number}: {len(row_data)} cells")
			return row_data
			
		except HttpError as e:
			self.logger.error(f"API error when reading row {row_number}: {e}")
			raise
		except Exception as e:
			self.logger.error(f"Error reading row {row_number}: {e}")
			raise

	def update_cell(self, spreadsheet_id: str, sheet_name: str, 
					 cell: str, value: Any) -> None:
		"""
		Update a single cell in a sheet.
		
		Args:
			spreadsheet_id (str): The ID of the spreadsheet
			sheet_name (str): Name of the sheet to write to
			cell (str): Cell reference (e.g., 'A1')
			value (Any): Value to write
		"""
		range_name = f"'{sheet_name}'!{cell}"
		try:
			body = {
				'values': [[value]]
			}
			
			self.sheets.values().update(
				spreadsheetId=spreadsheet_id,
				range=range_name,
				valueInputOption='RAW',
				body=body
			).execute()
			
			self.logger.debug(f"Updated cell {cell} in sheet '{sheet_name}'")
			
		except HttpError as e:
			self.logger.error(f"API error when updating cell {cell}: {e}")
			raise
		except Exception as e:
			self.logger.error(f"Error updating cell {cell}: {e}")
			raise

	def batch_update_cells(self, spreadsheet_id: str, sheet_name: str, 
							updates: Dict[str, Any]) -> None:
		"""
		Batch update multiple cells in a sheet.
		
		Args:
			spreadsheet_id (str): The ID of the spreadsheet
			sheet_name (str): Name of the sheet to write to
			updates (Dict[str, Any]): Dictionary mapping cell references to values
		"""
		if not updates:
			self.logger.warning("No updates to perform")
			return
			
		batch_data = []
		for cell, value in updates.items():
			batch_data.append({
				'range': f"'{sheet_name}'!{cell}",
				'values': [[value]]
			})
			
		body = {
			'valueInputOption': 'RAW',
			'data': batch_data
		}
		
		try:
			self.sheets.values().batchUpdate(
				spreadsheetId=spreadsheet_id,
				body=body
			).execute()
			
			self.logger.info(f"Batch updated {len(updates)} cells in sheet '{sheet_name}'")
			
		except HttpError as e:
			self.logger.error(f"API error during batch update: {e}")
			raise
		except Exception as e:
			self.logger.error(f"Error during batch update: {e}")
			raise

	def append_rows(self, spreadsheet_id: str, sheet_name: str, 
					 data: List[List[Any]]) -> None:
		"""
		Append rows to a sheet.
		
		Args:
			spreadsheet_id (str): The ID of the spreadsheet
			sheet_name (str): Name of the sheet to write to
			data (List[List[Any]]): List of rows to append
		"""
		if not data:
			self.logger.warning("No data to append")
			return
			
		range_name = f"'{sheet_name}'!A1"
		body = {
			'values': data
		}
		
		try:
			self.sheets.values().append(
				spreadsheetId=spreadsheet_id,
				range=range_name,
				valueInputOption='RAW',
				insertDataOption='INSERT_ROWS',
				body=body
			).execute()
			
			self.logger.info(f"Appended {len(data)} rows to sheet '{sheet_name}'")
			
		except HttpError as e:
			self.logger.error(f"API error when appending rows: {e}")
			raise
		except Exception as e:
			self.logger.error(f"Error appending rows: {e}")
			raise

	def clear_sheet(self, spreadsheet_id: str, sheet_name: str) -> None:
		"""
		Clear all data from a sheet.
		
		Args:
			spreadsheet_id (str): The ID of the spreadsheet
			sheet_name (str): Name of the sheet to clear
		"""
		try:
			self.sheets.values().clear(
				spreadsheetId=spreadsheet_id,
				range=f"'{sheet_name}'!A1:Z"
			).execute()
			
			self.logger.info(f"Cleared sheet '{sheet_name}'")
			
		except HttpError as e:
			self.logger.error(f"API error when clearing sheet: {e}")
			raise
		except Exception as e:
			self.logger.error(f"Error clearing sheet: {e}")
			raise

	def get_all_sheet_data(self, spreadsheet_id: str, sheet_name: str) -> List[List[Any]]:
		"""
		Get all data from a sheet.
		
		Args:
			spreadsheet_id (str): The ID of the spreadsheet
			sheet_name (str): Name of the sheet to read from
			
		Returns:
			List[List[Any]]: All data from the sheet
		"""
		try:
			result = self.sheets.values().get(
				spreadsheetId=spreadsheet_id,
				range=f"'{sheet_name}'!A1:Z"
			).execute()
			
			values = result.get('values', [])
			self.logger.info(f"Read {len(values)} rows from sheet '{sheet_name}'")
			return values
			
		except HttpError as e:
			self.logger.error(f"API error when reading full sheet: {e}")
			raise
		except Exception as e:
			self.logger.error(f"Error reading full sheet: {e}")
			raise

In [2]:
import logging
import asyncio
import json
import time
import random
from typing import Dict, List, Any, Optional, Tuple
from dataclasses import dataclass

@dataclass
class PipelineConfig:
    """Configuration for the processing pipeline"""
    sheet_url: str
    credentials_file: str
    read_sheet_name: str
    write_sheet_name: str
    metadata_column: str = "V"
    start_row: int = 2
    end_row: Optional[int] = None
    
    # Rate limiting settings
    batch_size: int = 5  # Reduced batch size
    sleep_between_batches: float = 3.0  # Increased sleep time
    max_requests_per_minute: int = 50  # Keep under the 60 limit
    read_delay: float = 1.5  # Time between read operations
    write_delay: float = 2.0  # Time between write operations
    
    # Retry settings
    max_retries: int = 5
    initial_retry_delay: float = 1.0
    max_retry_delay: float = 60.0
    retry_multiplier: float = 2.0
    
    # Filter settings
    threshold: int = 2

class RateLimiter:
    """Rate limiter for API calls"""
    
    def __init__(self, max_calls: int, time_period: float = 60.0):
        """
        Initialize rate limiter
        
        Args:
            max_calls: Maximum number of calls allowed in the time period
            time_period: Time period in seconds (default: 60 seconds = 1 minute)
        """
        self.max_calls = max_calls
        self.time_period = time_period
        self.call_timestamps = []
        self.logger = logging.getLogger(__name__)
        
    async def wait_if_needed(self):
        """
        Wait if we're at the rate limit
        """
        now = time.time()
        
        # Remove timestamps that are outside the time window
        self.call_timestamps = [ts for ts in self.call_timestamps 
                               if now - ts < self.time_period]
        
        # If we're at the limit, wait until we can make another call
        if len(self.call_timestamps) >= self.max_calls:
            oldest_timestamp = min(self.call_timestamps)
            wait_time = oldest_timestamp + self.time_period - now
            if wait_time > 0:
                self.logger.info(f"Rate limit reached, waiting for {wait_time:.2f} seconds")
                await asyncio.sleep(wait_time + 0.1)  # Add a small buffer
                
        # Record this call
        self.call_timestamps.append(time.time())

class MetadataProcessor:
    """Class for parsing and processing metadata from column V"""
    
    @staticmethod
    def parse_metadata(metadata_str: str) -> Dict[str, int]:
        """Parse metadata string in format 'KEY1=VAL1,KEY2=VAL2,...'"""
        if not metadata_str:
            return {}
            
        result = {}
        try:
            pairs = metadata_str.split(',')
            for pair in pairs:
                if '=' in pair:
                    key, value = pair.split('=', 1)
                    result[key.strip()] = int(value.strip())
                else:
                    logging.warning(f"Malformed metadata pair: {pair}")
        except Exception as e:
            logging.error(f"Error parsing metadata '{metadata_str}': {e}")
            
        return result
    
    @staticmethod
    def calculate_sum(metadata: Dict[str, int]) -> int:
        """Calculate the sum of all metadata values"""
        return sum(metadata.values())
    
    @staticmethod
    def format_metadata(metadata: Dict[str, int]) -> str:
        """Format metadata dictionary as a readable string"""
        if not metadata:
            return "No metadata available"
            
        formatted_pairs = [f"{key}: {value}" for key, value in metadata.items()]
        return "\n".join(formatted_pairs)

class GoogleSheetsRateLimitedManager:
    """Enhanced Google Sheets Manager with rate limiting and retries"""
    
    def __init__(self, credentials_file: str, config: PipelineConfig):
        """Initialize with credentials and rate limiting config"""
        self.logger = logging.getLogger(__name__)
        self.config = config
        
        # Initialize underlying manager
        self.manager = GoogleSheetsManager(credentials_file)
        
        # Set up rate limiters
        self.read_limiter = RateLimiter(config.max_requests_per_minute)
        self.write_limiter = RateLimiter(config.max_requests_per_minute)
        
    def extract_spreadsheet_id(self, sheet_url: str) -> str:
        """Extract spreadsheet ID from URL"""
        return self.manager.extract_spreadsheet_id(sheet_url)
        
    async def get_sheet_id(self, spreadsheet_id: str, sheet_name: str) -> int:
        """Get sheet ID with rate limiting and retries"""
        for attempt in range(self.config.max_retries):
            try:
                await self.read_limiter.wait_if_needed()
                return self.manager.get_sheet_id(spreadsheet_id, sheet_name)
                
            except Exception as e:
                if self._is_rate_limit_error(e):
                    wait_time = self._calculate_retry_delay(attempt)
                    self.logger.warning(f"Rate limit hit, retrying in {wait_time:.2f}s: {e}")
                    await asyncio.sleep(wait_time)
                elif attempt < self.config.max_retries - 1:
                    wait_time = self._calculate_retry_delay(attempt)
                    self.logger.error(f"Error getting sheet ID, retrying in {wait_time:.2f}s: {e}")
                    await asyncio.sleep(wait_time)
                else:
                    self.logger.error(f"Failed to get sheet ID after {attempt+1} attempts: {e}")
                    raise
                    
        raise Exception(f"Failed to get sheet ID after {self.config.max_retries} attempts")
        
    async def read_multiple_rows(self, spreadsheet_id: str, sheet_name: str, 
                               start_row: int, num_rows: int) -> List[List[Any]]:
        """Read multiple rows at once to reduce API calls"""
        for attempt in range(self.config.max_retries):
            try:
                await self.read_limiter.wait_if_needed()
                end_row = start_row + num_rows - 1
                range_name = f"A{start_row}:Z{end_row}"
                
                result = self.manager.read_range(spreadsheet_id, sheet_name, range_name)
                
                # Add a delay after successful read
                await asyncio.sleep(self.config.read_delay)
                
                return result
                
            except Exception as e:
                if self._is_rate_limit_error(e):
                    wait_time = self._calculate_retry_delay(attempt)
                    self.logger.warning(f"Rate limit hit, retrying in {wait_time:.2f}s: {e}")
                    await asyncio.sleep(wait_time)
                elif attempt < self.config.max_retries - 1:
                    wait_time = self._calculate_retry_delay(attempt)
                    self.logger.error(f"Error reading rows, retrying in {wait_time:.2f}s: {e}")
                    await asyncio.sleep(wait_time)
                else:
                    self.logger.error(f"Failed to read rows after {attempt+1} attempts: {e}")
                    raise
                    
        raise Exception(f"Failed to read rows after {self.config.max_retries} attempts")
        
    async def read_metadata_column(self, spreadsheet_id: str, sheet_name: str,
                                 start_row: int, num_rows: int, 
                                 metadata_column: str) -> List[str]:
        """Read just the metadata column for multiple rows"""
        for attempt in range(self.config.max_retries):
            try:
                await self.read_limiter.wait_if_needed()
                end_row = start_row + num_rows - 1
                range_name = f"{metadata_column}{start_row}:{metadata_column}{end_row}"
                
                result = self.manager.read_range(spreadsheet_id, sheet_name, range_name)
                
                # Flatten the result
                metadata_values = [row[0] if row else "" for row in result]
                
                # Add a delay after successful read
                await asyncio.sleep(self.config.read_delay)
                
                return metadata_values
                
            except Exception as e:
                if self._is_rate_limit_error(e):
                    wait_time = self._calculate_retry_delay(attempt)
                    self.logger.warning(f"Rate limit hit, retrying in {wait_time:.2f}s: {e}")
                    await asyncio.sleep(wait_time)
                elif attempt < self.config.max_retries - 1:
                    wait_time = self._calculate_retry_delay(attempt)
                    self.logger.error(f"Error reading metadata, retrying in {wait_time:.2f}s: {e}")
                    await asyncio.sleep(wait_time)
                else:
                    self.logger.error(f"Failed to read metadata after {attempt+1} attempts: {e}")
                    raise
                    
        raise Exception(f"Failed to read metadata after {self.config.max_retries} attempts")
        
    async def batch_update_cells(self, spreadsheet_id: str, sheet_name: str, 
                               updates: Dict[str, Any]) -> None:
        """Batch update cells with rate limiting and retries"""
        if not updates:
            return
            
        for attempt in range(self.config.max_retries):
            try:
                await self.write_limiter.wait_if_needed()
                
                self.manager.batch_update_cells(spreadsheet_id, sheet_name, updates)
                
                # Add a delay after successful write
                await asyncio.sleep(self.config.write_delay)
                
                return
                
            except Exception as e:
                if self._is_rate_limit_error(e):
                    wait_time = self._calculate_retry_delay(attempt)
                    self.logger.warning(f"Rate limit hit, retrying in {wait_time:.2f}s: {e}")
                    await asyncio.sleep(wait_time)
                elif attempt < self.config.max_retries - 1:
                    wait_time = self._calculate_retry_delay(attempt)
                    self.logger.error(f"Error updating cells, retrying in {wait_time:.2f}s: {e}")
                    await asyncio.sleep(wait_time)
                else:
                    self.logger.error(f"Failed to update cells after {attempt+1} attempts: {e}")
                    raise
                    
        raise Exception(f"Failed to update cells after {self.config.max_retries} attempts")
        
    async def clear_sheet(self, spreadsheet_id: str, sheet_name: str) -> None:
        """Clear sheet with rate limiting and retries"""
        for attempt in range(self.config.max_retries):
            try:
                await self.write_limiter.wait_if_needed()
                
                self.manager.clear_sheet(spreadsheet_id, sheet_name)
                
                # Add a delay after successful write
                await asyncio.sleep(self.config.write_delay)
                
                return
                
            except Exception as e:
                if self._is_rate_limit_error(e):
                    wait_time = self._calculate_retry_delay(attempt)
                    self.logger.warning(f"Rate limit hit, retrying in {wait_time:.2f}s: {e}")
                    await asyncio.sleep(wait_time)
                elif attempt < self.config.max_retries - 1:
                    wait_time = self._calculate_retry_delay(attempt)
                    self.logger.error(f"Error clearing sheet, retrying in {wait_time:.2f}s: {e}")
                    await asyncio.sleep(wait_time)
                else:
                    self.logger.error(f"Failed to clear sheet after {attempt+1} attempts: {e}")
                    raise
                    
        raise Exception(f"Failed to clear sheet after {self.config.max_retries} attempts")
        
    async def get_row_count(self, spreadsheet_id: str, sheet_name: str) -> int:
        """Get the total number of rows in a sheet"""
        for attempt in range(self.config.max_retries):
            try:
                await self.read_limiter.wait_if_needed()
                
                # Get sheet data but only request one column to minimize data transfer
                range_name = f"A:A"
                result = self.manager.read_range(spreadsheet_id, sheet_name, range_name)
                
                # Add a delay after successful read
                await asyncio.sleep(self.config.read_delay)
                
                return len(result) if result else 0
                
            except Exception as e:
                if self._is_rate_limit_error(e):
                    wait_time = self._calculate_retry_delay(attempt)
                    self.logger.warning(f"Rate limit hit, retrying in {wait_time:.2f}s: {e}")
                    await asyncio.sleep(wait_time)
                elif attempt < self.config.max_retries - 1:
                    wait_time = self._calculate_retry_delay(attempt)
                    self.logger.error(f"Error getting row count, retrying in {wait_time:.2f}s: {e}")
                    await asyncio.sleep(wait_time)
                else:
                    self.logger.error(f"Failed to get row count after {attempt+1} attempts: {e}")
                    raise
                    
        raise Exception(f"Failed to get row count after {self.config.max_retries} attempts")
    
    def _is_rate_limit_error(self, error: Exception) -> bool:
        """Check if an error is a rate limit error"""
        error_str = str(error).lower()
        return "429" in error_str or "quota" in error_str or "rate" in error_str
        
    def _calculate_retry_delay(self, attempt: int) -> float:
        """Calculate exponential backoff with jitter for retries"""
        delay = min(
            self.config.max_retry_delay,
            self.config.initial_retry_delay * (self.config.retry_multiplier ** attempt)
        )
        # Add jitter to avoid thundering herd problem
        jitter = random.uniform(0, 0.1 * delay)
        return delay + jitter

class FilterPipeline:
    """Pipeline for processing and filtering Google Sheet data with rate limiting"""
    
    def __init__(self, config: PipelineConfig):
        """Initialize the processing pipeline"""
        self.config = config
        self.logger = logging.getLogger(__name__)
        self.sheets_manager = GoogleSheetsRateLimitedManager(config.credentials_file, config)
        self.metadata_processor = MetadataProcessor()
        self.next_write_row = config.start_row
        
    async def initialize(self) -> None:
        """Set up the pipeline and validate configuration"""
        self.logger.info("Initializing processing pipeline")
        try:
            self.spreadsheet_id = self.sheets_manager.extract_spreadsheet_id(self.config.sheet_url)
            
            # Validate sheets exist
            await self.sheets_manager.get_sheet_id(self.spreadsheet_id, self.config.read_sheet_name)
            await self.sheets_manager.get_sheet_id(self.spreadsheet_id, self.config.write_sheet_name)
            
            # Clear the output sheet before starting
            await self.sheets_manager.clear_sheet(self.spreadsheet_id, self.config.write_sheet_name)
            
            self.logger.info("Pipeline initialization successful")
            
        except Exception as e:
            self.logger.error(f"Failed to initialize pipeline: {e}")
            raise
    
    async def get_row_count(self) -> int:
        """Get the number of rows to process"""
        try:
            # Get the total number of rows in the sheet
            total_rows = await self.sheets_manager.get_row_count(
                self.spreadsheet_id, self.config.read_sheet_name)
            
            # Adjust for start_row (which is 1-indexed)
            available_rows = max(0, total_rows - (self.config.start_row - 1))
            
            # If end_row is specified, use it to limit the number of rows
            if self.config.end_row is not None:
                available_rows = min(
                    available_rows,
                    max(0, self.config.end_row - (self.config.start_row - 1))
                )
                
            self.logger.info(f"Found {available_rows} rows to process")
            return available_rows
            
        except Exception as e:
            self.logger.error(f"Error determining row count: {e}")
            return 0
            
    async def process_batch(self, start_idx: int, batch_size: int, total_rows: int) -> List[Dict[str, Any]]:
        """Process a batch of rows efficiently"""
        # Calculate actual batch size (in case we're at the end)
        actual_batch_size = min(batch_size, total_rows - start_idx)
        
        if actual_batch_size <= 0:
            return []
            
        self.logger.info(f"Processing batch of {actual_batch_size} rows starting at index {start_idx}")
        
        try:
            # Get the start row number (1-indexed)
            start_row = self.config.start_row + start_idx
            
            # OPTIMIZATION: Read all row data at once
            all_rows_data = await self.sheets_manager.read_multiple_rows(
                self.spreadsheet_id,
                self.config.read_sheet_name,
                start_row,
                actual_batch_size
            )
            
            # OPTIMIZATION: Read metadata column separately if needed
            metadata_col_idx = ord(self.config.metadata_column.upper()) - ord('A')
            
            batch_results = []
            
            # Process each row
            for i, row_data in enumerate(all_rows_data):
                row_num = start_row + i
                
                try:
                    # Get metadata from the row data
                    metadata_str = ""
                    if len(row_data) > metadata_col_idx:
                        metadata_str = row_data[metadata_col_idx]
                    
                    # Parse metadata
                    metadata = self.metadata_processor.parse_metadata(metadata_str)
                    
                    # Calculate sum
                    metadata_sum = self.metadata_processor.calculate_sum(metadata)
                    
                    # Determine if we should keep this row (sum <= threshold)
                    keep_row = metadata_sum <= self.config.threshold
                    
                    # Format for human-readable output
                    formatted_metadata = self.metadata_processor.format_metadata(metadata)
                    
                    # Log the decision
                    if keep_row:
                        self.logger.info(f"Row {row_num}: Sum = {metadata_sum} <= {self.config.threshold}, KEEPING")
                    else:
                        self.logger.info(f"Row {row_num}: Sum = {metadata_sum} > {self.config.threshold}, SKIPPING")
                    
                    batch_results.append({
                        "row": row_num,
                        "row_data": row_data,
                        "metadata": metadata,
                        "metadata_sum": metadata_sum,
                        "keep_row": keep_row,
                        "formatted_metadata": formatted_metadata,
                        "success": True
                    })
                    
                except Exception as e:
                    self.logger.error(f"Error processing row {row_num}: {e}")
                    batch_results.append({
                        "row": row_num,
                        "keep_row": False,
                        "success": False,
                        "error": str(e)
                    })
            
            # Calculate statistics
            filtered_count = sum(1 for result in batch_results if result.get('keep_row', False))
            success_count = sum(1 for result in batch_results if result.get('success', False))
            
            self.logger.info(f"Batch processing complete: {success_count}/{len(batch_results)} successful, "
                             f"{filtered_count}/{len(batch_results)} rows kept")
            
            return batch_results
            
        except Exception as e:
            self.logger.error(f"Error processing batch: {e}")
            return []
    
    async def write_results(self, results: List[Dict[str, Any]]) -> int:
        """Write processing results back to the output sheet"""
        if not results:
            return 0
        
        # Filter to keep only rows that meet our criteria
        rows_to_write = [r for r in results if r.get('success', False) and r.get('keep_row', False)]
        
        if not rows_to_write:
            self.logger.info("No rows met the criteria in this batch")
            return 0
        
        try:
            # Prepare data for writing in an efficient format
            updates = {}
            for result in rows_to_write:
                row_data = result['row_data']
                output_row = self.next_write_row
                
                # Create entries for each cell in the row
                for col_idx, cell_value in enumerate(row_data):
                    col_letter = chr(ord('A') + col_idx)
                    cell_ref = f"{col_letter}{output_row}"
                    updates[cell_ref] = cell_value
                
                self.next_write_row += 1
            
            # Write to sheet
            if updates:
                await self.sheets_manager.batch_update_cells(
                    self.spreadsheet_id,
                    self.config.write_sheet_name,
                    updates
                )
            
            self.logger.info(f"Wrote {len(rows_to_write)} rows to output sheet")
            return len(rows_to_write)
            
        except Exception as e:
            self.logger.error(f"Error writing results to sheet: {e}")
            return 0
                
    async def run(self) -> Dict[str, Any]:
        """Run the full processing pipeline"""
        self.logger.info("Starting processing pipeline")
        start_time = asyncio.get_event_loop().time()
        
        try:
            # Initialize the pipeline
            await self.initialize()
            
            # Get the total number of rows to process
            total_rows = await self.get_row_count()
            if total_rows == 0:
                self.logger.warning("No rows to process")
                return {"success": True, "rows_processed": 0, "elapsed_time": 0}
                
            # Process in batches
            processed_rows = 0
            filtered_rows = 0
            stats = {
                "total_rows": total_rows,
                "successful_rows": 0,
                "failed_rows": 0,
                "filtered_rows": 0  # Rows that met our criteria
            }
            
            while processed_rows < total_rows:
                # Process a batch
                batch_results = await self.process_batch(
                    processed_rows, 
                    self.config.batch_size, 
                    total_rows
                )
                
                # Update statistics
                stats["successful_rows"] += sum(1 for r in batch_results if r.get('success', False))
                stats["failed_rows"] += sum(1 for r in batch_results if not r.get('success', False))
                
                # Write results to the output sheet and count rows written
                rows_written = await self.write_results(batch_results)
                stats["filtered_rows"] += rows_written
                
                # Update progress
                processed_rows += len(batch_results)
                progress_pct = (processed_rows / total_rows) * 100
                self.logger.info(f"Progress: {processed_rows}/{total_rows} ({progress_pct:.1f}%)")
                
                # Sleep between batches to avoid API rate limits
                if processed_rows < total_rows:
                    sleep_time = self.config.sleep_between_batches
                    self.logger.info(f"Sleeping for {sleep_time}s before next batch")
                    await asyncio.sleep(sleep_time)
            
            # Calculate elapsed time
            elapsed_time = asyncio.get_event_loop().time() - start_time
            stats["elapsed_time"] = elapsed_time
            stats["rows_per_second"] = total_rows / elapsed_time if elapsed_time > 0 else 0
            
            self.logger.info(f"Pipeline completed in {elapsed_time:.2f} seconds")
            self.logger.info(f"Processed {stats['successful_rows']} rows successfully, {stats['failed_rows']} failures")
            self.logger.info(f"Found {stats['filtered_rows']} rows with sum <= {self.config.threshold}")
            
            return {"success": True, **stats}
            
        except Exception as e:
            elapsed_time = asyncio.get_event_loop().time() - start_time
            self.logger.error(f"Pipeline failed: {e}")
            return {"success": False, "error": str(e), "elapsed_time": elapsed_time}

async def main_async():
    """Async main function to run the pipeline"""
    # Configure logging
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
        handlers=[
            logging.StreamHandler(),
            logging.FileHandler('pipeline.log')
        ]
    )
    logger = logging.getLogger(__name__)
    
    try:
        # Load configuration from file
        config_data = ({			
			"sheet_url": "https://docs.google.com/spreadsheets/d/1zhDMUVl-75GiY-gIFc-jL9A2E3KVd9XpfV21D3pMVXo/edit?gid=855023356#gid=855023356",
			"credentials_file": "data/url-to-email-445616-cebe4868914f.json",
			"read_sheet_name": "Sheet1",
			"write_sheet_name": "Sheet3",
			"metadata_column": "V",
			"start_row": 2,
			"end_row": None,
			"batch_size": 5,
			"sleep_between_batches": 3.0,
			"max_requests_per_minute": 50,
			"read_delay": 1.5,
			"write_delay": 2.0,
			
			"max_retries": 5,
			"initial_retry_delay": 1.0,
			"max_retry_delay": 60.0,
			"retry_multiplier": 2.0,
			
			"threshold": 2
		})
        
        # Create pipeline config
        config = PipelineConfig(**config_data)
        
        # Create and run the pipeline
        pipeline = FilterPipeline(config)
        results = await pipeline.run()
        
        # Print summary
        if results["success"]:
            logger.info("Pipeline completed successfully!")
            logger.info(f"Processed {results['total_rows']} rows in {results['elapsed_time']:.2f} seconds")
            logger.info(f"Found {results['filtered_rows']} rows with sum <= {config.threshold}")
            logger.info(f"Success rate: {results['successful_rows'] / results['total_rows'] * 100:.1f}%")
        else:
            logger.error(f"Pipeline failed: {results.get('error', 'Unknown error')}")
            
    except Exception as e:
        logger.error(f"Error in main process: {e}", exc_info=True)

In [3]:
await main_async()

2025-07-22 11:18:38,908 - googleapiclient.discovery_cache - INFO - file_cache is only supported with oauth2client<4.0.0
2025-07-22 11:18:38,993 - __main__ - INFO - Successfully authenticated with Google Sheets API
2025-07-22 11:18:38,994 - __main__ - INFO - Starting processing pipeline
2025-07-22 11:18:38,994 - __main__ - INFO - Initializing processing pipeline
2025-07-22 11:18:40,592 - __main__ - INFO - Cleared sheet 'Sheet3'
2025-07-22 11:18:42,593 - __main__ - INFO - Pipeline initialization successful
2025-07-22 11:18:43,032 - __main__ - INFO - Read 1001 rows from 'Sheet1'!A:A
2025-07-22 11:18:44,533 - __main__ - INFO - Found 1000 rows to process
2025-07-22 11:18:44,534 - __main__ - INFO - Processing batch of 5 rows starting at index 0
2025-07-22 11:18:44,985 - __main__ - INFO - Read 5 rows from 'Sheet1'!A2:Z6
2025-07-22 11:18:46,486 - __main__ - INFO - Row 2: Sum = 0 <= 2, KEEPING
2025-07-22 11:18:46,486 - __main__ - INFO - Row 3: Sum = 2 <= 2, KEEPING
2025-07-22 11:18:46,487 - __m